# Multi-Modal Distributed Extraction Worker 16/48
Deterministic distributed worker 16 of 48 with TransNetV2, YOLO-World, DAM-3B, Vietnamese OCR, SigLIP2 embeddings, and atomic Rclone sync.

In [ ]:
# 1. Safely step out to /kaggle/working, wipe old repo, and clone fresh
import os, sys
from pathlib import Path

if Path("/kaggle/working").exists():
    os.chdir("/kaggle/working")
    %cd /kaggle/working

!rm -rf /kaggle/working/AIC-2026
!git clone -b feature/dam-text-extraction https://github.com/AIVIETNAM-AIO-Dewey/AIC-2026.git /kaggle/working/AIC-2026
%cd /kaggle/working/AIC-2026

if str(Path.cwd() / "src") not in sys.path:
    sys.path.insert(0, str(Path.cwd() / "src"))

print(f"✓ Current working directory: {Path.cwd()}")
!git log -1 --oneline


In [ ]:
import os, re, subprocess
from pathlib import Path
from kaggle_secrets import UserSecretsClient

# 1. Install rclone binary
!apt-get update -qq && apt-get install -y -qq rclone

# 2. Write rclone config to all canonical paths & set RCLONE_CONFIG env var
try:
    raw_secret = UserSecretsClient().get_secret("RCLONE_CONFIG_GDRIVE").strip()
    if not raw_secret.startswith("["):
        raw_secret = "[gdrive] " + raw_secret
    if chr(10) not in raw_secret:
        raw_secret = raw_secret.replace("[gdrive]", "[gdrive]" + chr(10))
        raw_secret = re.sub(r"\s+(type\s*=)", chr(10) + r"\1", raw_secret)
        raw_secret = re.sub(r"\s+(scope\s*=)", chr(10) + r"\1", raw_secret)
        raw_secret = re.sub(r"\s+(token\s*=)", chr(10) + r"\1", raw_secret)
        raw_secret = re.sub(r"\s+(client_id\s*=)", chr(10) + r"\1", raw_secret)
        raw_secret = re.sub(r"\s+(client_secret\s*=)", chr(10) + r"\1", raw_secret)
    config_paths = [
        "/root/.config/rclone/rclone.conf",
        "/root/.rclone.conf",
        "/kaggle/working/.rclone.conf",
        "/tmp/rclone.conf",
    ]
    for path_str in config_paths:
        p = Path(path_str)
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text(raw_secret, encoding="utf-8")
    os.environ["RCLONE_CONFIG"] = "/root/.config/rclone/rclone.conf"
    print("✓ rclone.conf written and RCLONE_CONFIG exported!")
    t = subprocess.run(["rclone", "listremotes", "--config", "/root/.config/rclone/rclone.conf"], capture_output=True, text=True)
    print("✓ Configured remotes:", t.stdout.strip())
except Exception as exc:
    print(f"❌ Error: {exc}")


In [ ]:
!python -m pip install --quiet --no-deps -r requirements/kaggle.txt
!python -m pip install --quiet easyocr


In [ ]:
WORKER_ID = 16
NUM_WORKERS = 48
VIDEOS_ROOT = "/kaggle/input"
RCLONE_DEST = "gdrive:AIC_HCM_BATCH_1/artifacts/"

# 1-second dry run verification
!python scripts/run_dam_batch.py \
  --worker-id {WORKER_ID} \
  --num-workers {NUM_WORKERS} \
  --videos-root {VIDEOS_ROOT} \
  --frame-extractor transnetv2 \
  --detector yolo-world \
  --detector-model yolov8x-worldv2.pt \
  --enable-ocr \
  --enable-siglip \
  --rclone-dest {RCLONE_DEST} \
  --dry-run


In [ ]:
WORKER_ID = 16
NUM_WORKERS = 48
VIDEOS_ROOT = "/kaggle/input"
RCLONE_DEST = "gdrive:AIC_HCM_BATCH_1/artifacts/"

# Launch full automated multi-modal batch run
!python scripts/run_dam_batch.py \
  --worker-id {WORKER_ID} \
  --num-workers {NUM_WORKERS} \
  --videos-root {VIDEOS_ROOT} \
  --frame-extractor transnetv2 \
  --detector yolo-world \
  --detector-model yolov8x-worldv2.pt \
  --enable-ocr \
  --enable-siglip \
  --rclone-dest {RCLONE_DEST} \
  --output-root /kaggle/working/aic2026-artifacts \
  --cache-root /kaggle/working/aic2026-model-cache \
  --device cuda
